<img src='../OUTILS/bandeau_MF.png' align='right' width='100%'/>

# <div style='background-color: #27ae60; color: white; padding: 20px; border-radius: 10px; text-align: center;'>🌍🛰️ Manipulation de données satellitaires - Format NetCDF</div>

---

## 🎯 **Objectifs du TP**


## 🔎  Analyser un fichier NetCDF 
## 🖼️  Produire une image à partir d'un canal 
## 🎚️  Modifier la dynamique des valeurs 
## 📍  Extraire une valeur ponctuelle 

---

## 📑 **Table des matières**

| Section | Titre |
|---------|-------|
| 🔗 | [**1.** 🔎 Visualisation du contenu du NetCDF](#1-visualisation-du-contenu) |
| 🔗 | [**2.** 🖼️ Production d'une image TIF (canal VIS006)](#2-production-dune-image-tif-canal-vis006) |
| 🔗 | [**3.** 🌡️ Production d'une image TIF (canal IR_105)](#3-production-dune-image-tif-canal-ir_105) |
| 🔗 | [**4.** 📍 Extraction de valeurs ponctuelles](#4-extraction-de-valeurs-ponctuelles) | 
| 🔗 | [**5.** 📍 Extraction de valeurs ponctuelles à partir des TIF](#5-extraction-de-valeurs-ponctuelles) |
| 🔗 | [**6.** 📍 Exemples réalisés à partir d'extractions](6-exemples) |

---

## 🧰 **Workflow : lignes de commandes bash (GDAL & ImageMagick)**

Ce TP utilise en grande partie la bibliothèque **GDAL** (*Geospatial Data Abstraction Library*), un outil puissant pour manipuler les données issues de satellites météorologiques.

Les fichiers utilisés sont au format **NetCDF**, issus de la production opérationnelle de Météo-France.

---

## 📂 **Fichiers utilisés**

| Fichier | Résolution | Description |
|---------|------------|-------------|
| `Mmultic500mNC4_mtgi1_202604221200.nc` | 500 m | 2 canaux |
| `Mmultic1kmNC4_mtgi1_202604221200.nc` | 1 km | 10 canaux |
| `Mmultic2kmNC4_mtgi1_202604221200.nc` | 2 km | 16 canaux |

📁 **Répertoire** : `/stockage/DATA/NetCDF/20260422`

> 💡 *Ces fichiers multicanaux sont produits par le Centre de Météorologie Spatiale (CMS) de Météo-France pour les satellites géostationnaires.*

<div class="alert alert-info" role="alert">
<h3> ⚙️ Initialisation de l'environnement</h3>
Importation des librairies nécessaires et configuration des chemins d'accès.
</div>

In [ ]:
from datetime import datetime
import sys
import os
from osgeo import gdal
import subprocess
from IPython.display import display,HTML
import glob
from IPython.display import Image as IPImage  # Alias pour IPython
from PIL import Image as PILImage              # Alias pour PIL
os.environ['PATH'] = f"/opt/conda/env_MF_teledetection/bin:{os.environ['PATH']}" 
os.environ['PATH'] = f"~/.conda/envs/env_MF_teledetection/bin:{os.environ['PATH']}"
os.environ['GDAL_DATA'] = '/opt/conda/env_MF_teledetection/share/gdal'
os.environ['PROJ_LIB'] = '/opt/conda/env_MF_teledetection/share/proj'

## 📡 **Rappel des différents canaux**

<div align="center">
<a href="../DOCS/mtg_fci_synthese_produits.jpg" target="_blank">
    <img src='../DOCS/mtg_fci_synthese_produits.jpg' alt='Canaux MTG' width='900px'>
</a>
</div>

In [ ]:
RepSource="/stockage/DATA/20260614/*"
!mkdir -p RESULTS
output = 'RESULTS'
!mkdir -p RESULTS/tmp

print("📁 Fichiers configurés :")
print(f"   - Chunks Meteosat-12 : {RepSource}")

<div id="1-visualisation-du-contenu"></div>
<div class="alert alert-info alert-success">
<h3 id="1"> 1️⃣ - 🔎 Visualisation du contenu du NetCDF</h3>
</div>

La commande `gdalinfo` permet d'afficher les métadonnées d'un fichier NetCDF. Dans Jupyter, on la précède de `!`.

In [ ]:
!ls -ltr {RepSource} | wc -l

In [ ]:
!ls -ltr {RepSource} | head -n 3 

### Regardons le contenu d'un fichier Chunk

In [ ]:
chunk1="/stockage/DATA/20260614/W_XX-EUMETSAT-Darmstadt,IMG+SAT,MTI1+FCI-1C-RRAD-FDHSI-FD--CHK-BODY---NC4E_C_EUMT_20260614120915_IDPFI_OPE_20260614120629_20260614120726_N__O_0073_0028.nc"
!gdalinfo {chunk1}

## 📡 Décryptage du fichier netCDF FCI (MTG)

| Information | Valeur | Signification |
|-------------|--------|----------------|
| 🏷️ **Format** | netCDF (NC4) | Données scientifiques multicouches, non géoréférencées directement |​
| 🛰️ **Satellite / Instrument** | MTI1 (MTG) / FCI | Meteosat Third Generation, Flexible Combined Imager |
| 📊 **Niveau de produit** | 1C - RRAD | Radiances rectifiées, géolocalisées mais non projetées |
| 📐 **Dimensions réelles** | **300 × 11136** pixels par canal | 300 lignes de balayage, largeur du disque complet |
| 🌐 **Couverture géographique** | Lat : 18.48°–24.78°N<br>Lon : -79.64°–78.45°E | Vue partielle (haute latitude), pas le disque entier |
| 🕒 **Date / Heure** | 2026-06-14 12:00:00 UTC | Heure du slot (12h) |
| 🎯 **Projection** | GEOS (Géostationnaire) | Longitude nominale 0°, hauteur 35 785 834 m |
| 🌈 **Canaux disponibles** | 16 canaux (vis, nIR, IR, WV) | De 0.4 µm à 13.3 µm, dont le WV 6.3 & 7.3 µm |
| 🧩 **Sous-ensembles** | `/data/vis_04/measured/effective_radiance` | Chaque canal contient radiance, qualité, index, swath |
| 📏 **Type de données** | uint16 (radiances), uint8 (qualité) | Radiances en comptages numériques (non calibrées en °C) | 


Liste des canaux disponibles

In [ ]:
!gdalinfo {chunk1}  2>/dev/null | grep -E "FCI FDHSI"

In [ ]:
#!gdalinfo -nomd NETCDF:{chunk1}:/data/vis_06/measured/effective_radiance 
!gdalinfo NETCDF:{chunk1}:/data/vis_06/measured/effective_radiance 
#Remarque l'option -nomd est importante pour limiter le résultat au seul informations du canal recherché

## 📡 Décryptage du canal VIS06 (FCI – MTG)

| Information | Valeur | Signification |
|-------------|--------|----------------|
| 🏷️ **Format** | netCDF4 | Données scientifiques hiérarchisées (groupe `/data/vis_06/measured/`) |
| 📐 **Dimensions** | **11136 × 300** pixels | Largeur du disque complet (colonnes) × nombre de lignes d​u chunk |
| 🛰️ **Projection** | GEOS (Sweep Y) | Vue géostationnaire, satellite MTG, longitude nominale 0° |
| 📏 **Pixel Size** | **0.0000279436 rad**  | Résolution angulaire (radians) – très fine pour du visible |
| 📍 **Coordonnées** | x : -0.1556 à 0.1556 rad<br>y : 0.0556 à 0.0639 rad | Coordonnées projetées (en radians) correspondant aux angles de balayage |
| 📊 **Type de données** | **UInt16** (entier non signé 16 bits) | Comptages numériques bruts, à convertir en radiance |
| 📏 **Échelle & Offset** | scale = 0.00709761<br>offset = -1.4479115 | Formule : <br>**Radiance = count × scale + offset** |
| 🌈 **Grandeur physique** | `mW.m⁻².sr⁻¹.(cm⁻¹)⁻¹` | Radiance efficace (monochromatique) dans le visible |
| 🚫 **Valeur de remplissage** | 65535 | Pixels hors disque ou non valides |
| 🌐 **Couverture géographique** | Lat : 18.5°–24.8°N<br>Lon : -79.6°–78.5°E | Vue partielle (région Europe/Afrique/Atlantique) |
| 🕒 **Date / Heure** | 2026-06-14 12:00 UTC | Heure du slot (midi) | 

In [ ]:
!gdalinfo -nomd -mm NETCDF:{chunk1}:/data/ir_105/measured/effective_radiance |grep -E "Min/Max"

> 💡 *L'option `-mm` affiche les valeurs minimales et maximales trouvées dans le dataset.*

**Interprétation des valeurs** :

- `Computed Min/Max = 303.000, 2844.000` (pour VIS006)
- - `Computed Min/Max = -566.000, 3402.000` (pour IR_105)

⚠️ *Les réflectivités supérieures à 10 000 (>100 %) peuvent concerner des nuages glacés épais avec un soleil rasant.*

<div id="2-production-dune-image-tif-canal-vis006"></div>
<div class="alert alert-info alert-success">
<h3 id="2"> 2️⃣ - 🖼️ Production d'une image TIF à partir du canal VIS006</h3>
</div>

In [ ]:
cd ~/MF_DATA_MANIPULATION

In [ ]:
!mkdir -p RESULTS
output = 'RESULTS'
nom_fic = 'VIS006_mtg_20260614_1200'
redim = '2000'

print(f"📁 Dossier de sortie : {output}")
print(f"📏 Redimensionnement : {redim}")

### 🎨 **Étirement linéaire (contrast stretching, ou linear scaling)**

Nous allons transformer les valeurs physiques (16 bits) en nuances de gris (256 valeurs - 8 bits) avec `gdal_translate` et l'option `-scale` :

```
-scale entrée_min entrée_max sortie_min sortie_max
```
Le capteur FCI de MTG numérise les signaux sur 12 bits de dynamique. La valeur maximale de comptage numérique (DN) sera donc 4095 (2^12).</br>
Pour des raisons de compatibilité, la donnée est stockée sur 16 bits

In [ ]:
!gdal_translate -scale 0 4095 0 255 -ot byte NETCDF:{chunk1}:/data/vis_06/measured/effective_radiance $output/{nom_fic}.jpg 2>/dev/null
!convert -resize {redim} $output/{nom_fic}.jpg $output/{nom_fic}_{redim}.jpg 2>/dev/null
display(IPImage(filename=f"{output}/{nom_fic}_{redim}.jpg", width=1500)) 

### Pour visualiser plusieurs chunks, nous faisons une boucle sur chaque fichier 
-> Par exemple les chunk entre 30 et 39

In [ ]:
%%bash
rm -f RESULTS/tmp/*  # Nettoyage

for fic in /stockage/DATA/20260614/*_003[0-9].nc; do # Boucle
    echo "Extraction de $(basename ${fic})" 
    gdal_translate -q -scale 0 4095 0 255 -ot Byte "NETCDF:\"${fic}\":/data/vis_06/measured/effective_radiance" RESULTS/tmp/$(basename ${fic} .nc).tif 2>/dev/null || true  # Etirement
    convert -resize 1200 RESULTS/tmp/$(basename ${fic} .nc).tif RESULTS/$(basename ${fic} .nc).jpg 2>/dev/null #Redimmensionement 
done

In [ ]:
#Visualisation des chunks
fichiers = sorted(glob.glob("RESULTS/W_XX*.jpg"), reverse=True)
for fichier in fichiers:
    display(IPImage(filename=fichier, width=1500))  # Ajustez la taille 

### 🎚️ **Ajustement de la dynamique (windowing)**

En modifiant les bornes d'entrée, on peut rendre l'image plus lumineuse.

### Trouver les valeurs min et max de ce canal 

In [ ]:
!gdalinfo -nomd -mm NETCDF:{chunk1}:/data/vis_06/measured/effective_radiance |grep -E "Min/Max"

In [ ]:
%%bash
valeur_min_vis=303
valeur_max_vis=2844
rm -f RESULTS/tmp/*  # Nettoyage

for fic in /stockage/DATA/20260614/*_003[0-9].nc; do # Boucle
    #echo "Extraction de $(basename ${fic})" 
    gdal_translate -q -scale ${valeur_min_vis} ${valeur_max_vis} 0 255 -ot Byte "NETCDF:\"${fic}\":/data/vis_06/measured/effective_radiance" RESULTS/tmp/$(basename ${fic} .nc).tif 2>/dev/null || true  # Etirement
    convert -resize 1200 RESULTS/tmp/$(basename ${fic} .nc).tif RESULTS/$(basename ${fic} .nc).jpg 2>/dev/null #Redimmensionement 
done
echo "Extractions ok pour étirement -scale ${valeur_min_vis} ${valeur_max_vis} 0 255"

In [ ]:
#Visualisation des chunks
fichiers = sorted(glob.glob("RESULTS/W_XX*.jpg"), reverse=True)
for fichier in fichiers:
    display(IPImage(filename=fichier, width=1500))  # Ajustez la taille 

### <div style='background-color: #872459; color: white; padding: 10px; border-radius: 12px; text-align: left;'> ✏️ Exercice</div>

### 📐 **Correction Gamma (éclaircissement non linéaire)**

La correction Gamma permet d'éclaircir les zones sombres sans saturer les zones claires.</br>
Cet ajustement permet de modifier la luminosité d’une image de façon non linéaire.</br>
On parle aussi de « facteur de contraste »

| Gamma | Effet |
|-------|-------|
| < 1 | Assombrit l'image |
| = 1 | Aucun changement |
| > 1 | Éclaircit l'image |

> 💡 *GDAL utilise l'option `-exponent` avec comme valeur `1/gamma`.*

### Visualisation du plein disque vis06 brut

In [ ]:
!gdal_translate -scale 0 10000 0 255 -ot byte -outsize 2000 2000 /stockage/DATA/20260614/20260614_1200_vis006_16bits.tif {output}/20260614_1200_vis006_2000x2000.jpg #2>/dev/null
display(IPImage(filename=f"{output}/20260614_1200_vis006_2000x2000.jpg", width=700)) 

In [ ]:
### Application de la correction gamma et visualisation

In [ ]:
Gamma=1.5
ValGamma=1/Gamma
!gdal_translate -scale -exponent {ValGamma} -ot byte /stockage/DATA/20260614/20260614_1200_vis006_2000x2000.tif RESULTS/20260614_1200_vis006_2000x2000_gamma_{Gamma}.tif
!convert RESULTS/20260614_1200_vis006_2000x2000_gamma_{Gamma}.tif RESULTS/20260614_1200_vis006_2000x2000_gamma_{Gamma}.jpg
display(IPImage(filename=f"{output}/20260614_1200_vis006_2000x2000_gamma_{Gamma}.jpg", width=700)) 

In [ ]:
Gamma=2
ValGamma=1/Gamma

!gdal_translate -scale 0 10000 0 255 -exponent {ValGamma} -ot byte -outsize 2000 2000 /stockage/DATA/20260614/20260614_1200_vis006_16bits.tif {output}/20260614_1200_vis006_2000x2000_{Gamma}.jpg #2>/dev/null
display(IPImage(filename=f"{output}/20260614_1200_vis006_2000x2000_{Gamma}.jpg", width=700)) 

### <div style='background-color: #872459; color: white; padding: 10px; border-radius: 12px; text-align: left;'> ✏️ Exercice</div>

Eclaircir et assombrir l'image en jouant sur la dynamique et la valeur gamma 

In [ ]:
Gamma=
ValGamma=1/Gamma
!gdal_translate -scale 0 10000 0 255 -exponent {ValGamma} -ot byte -outsize 2000 2000 /stockage/DATA/20260614/20260614_1200_vis006_16bits.tif {output}/20260614_1200_vis006_2000x2000_{Gamma}.jpg #2>/dev/null
display(IPImage(filename=f"{output}/20260614_1200_vis006_2000x2000_{Gamma}.jpg", width=700)) 

> 💡 *Le produit opérationnel bénéficie de corrections supplémentaires : diffusion, angle solaire et parallaxe.*

<div id="3-production-dune-image-tif-canal-ir_105"></div>
<div class="alert alert-info alert-success">
<h3 id="3"> 3️⃣ - 🌡️ Production d'une image TIF à partir du canal IR_105</h3>
</div>

### 📊 **Spécificités du canal IR_105**

| Propriété | Valeur |
|-----------|--------|
| **Longueur d'onde** | 10.5 µm (infrarouge thermique) |
| **Grandeur** | toa_brightness_temperature |
| **Unités** | Kelvin (K) |
| **scale_factor** | 0.01 |
| **add_offset** | 273.15 |
| **Formule** | Température (K) = DN × 0.01 + 273.15 |

> 💡 *Les valeurs sont stockées en 16 bits en centièmes de degré Celsius : -50°C = -5000*

In [ ]:
!gdalinfo {chunk1} | grep ir_105

In [ ]:
!gdalinfo -nomd -mm NETCDF:{chunk1}:/data/ir_105/measured/effective_radiance |grep Min/Max

In [ ]:
nom_fic_ir = 'IR_105_mtg_20260614_1200'
!gdal_translate -scale 0 4095 0 255 -ot byte NETCDF:{chunk1}:/data/ir_105/measured/effective_radiance $output/{nom_fic_ir}.jpg 2>/dev/null
!convert -resize {redim} $output/{nom_fic_ir}.jpg $output/{nom_fic_ir}_{redim}.jpg 2>/dev/null
display(IPImage(filename=f"{output}/{nom_fic_ir}_{redim}.jpg", width=1500)) 

In [ ]:
%%bash
valeur_min_ir=303
valeur_max_ir=2844
rm -f RESULTS/tmp/*  # Nettoyage

for fic in /stockage/DATA/20260614/*_003[0-9].nc; do # Boucle
    #echo "Extraction de $(basename ${fic})" 
    gdal_translate -q -scale ${valeur_min_ir} ${valeur_max_ir} 0 255 -ot Byte "NETCDF:\"${fic}\":/data/ir_105/measured/effective_radiance" RESULTS/tmp/$(basename ${fic} .nc).tif 2>/dev/null || true  # Etirement
    convert -resize 1200 RESULTS/tmp/$(basename ${fic} .nc).tif RESULTS/IR_$(basename ${fic} .nc).jpg 2>/dev/null #Redimmensionement 
done
echo "Extractions ok pour étirement -scale ${valeur_min_ir} ${valeur_max_ir} 0 255"

In [ ]:
#Visualisation des chunks
fichiers = sorted(glob.glob("RESULTS/IR_W_XX*.jpg"), reverse=True)
for fichier in fichiers:
    display(IPImage(filename=fichier, width=1500))  # Ajustez la taille 

### Question : cette image vous parait-elle étrange ?

### <div style='background-color: #872459; color: white; padding: 10px; border-radius: 12px; text-align: left;'> ✏️ Exercice</div>

### Produire une image IR_105 avec :
- une autre dynamique de couleur

### 🌍 **Convention de représentation des températures**

| Région | Convention | Dynamique  ( `-scale` ) |
|--------|------------|----------------------|
| **Europe** | Nuages froids = blancs | `min max 255 0` |
| **USA** | Nuages chauds = blancs | `min max 0 255` |


---

## ✅ **Résumé des bonnes pratiques**

| Action | Commande / Option |
|--------|-------------------|
| Visualiser un NetCDF | `gdalinfo fichier.nc` |
| Extraire un canal | `NETCDF:\"fichier.nc\":canal` |
| Appliquer un étirement | `-scale min_in max_in min_out max_out` |
| Appliquer un gamma | `-exponent 1/gamma` |
| Extraire une valeur ponctuelle | `gdallocationinfo -wgs84 ... lon lat` |
| Interpolation cubique | `-r cubic` |

---

## 🔗 **Ressources complémentaires**

| Lien | Description |
|------|-------------|
| [Documentation GDAL](https://gdal.org) | Référence complète |
| [Galerie CMS](http://e-sat.cms.meteo.fr/GALERIE/index.html) | Galerie CMS |
| [Comparateur d'images](http://e-sat.cms.meteo.fr/GALERIE/slider4_vierge.html) | Outil de comparaison |

---

✅ **Fin du TP**